<a href="#Overview"></a>
# Overview
* <a href="#5330fd59-9c2d-4f45-b01c-0d91edb1e85d">NEUS 642 - Week 10 In-class Exercises</a>
  * <a href="#0bc2e68b-c3ef-4061-a613-f7e4f0d6cdb2">Background</a>
  * <a href="#a547739b-e2ac-49e0-ab7d-7541d1ed025d">Getting Started</a>
* <a href="#a12e372b-0b50-47cf-bb12-c9f4f606aff1">Exercise 1</a>
* <a href="#f84ebf64-6f5a-4fe9-8101-f885b152023e">Exercise 2</a>
  * <a href="#70c0aa42-a1c0-4e48-be53-c406565e64db">Exercise 3</a>
  * <a href="#13ca54dd-169d-4632-ab4e-60b4ab2aff9c">Exercise 4</a>

<a id="5330fd59-9c2d-4f45-b01c-0d91edb1e85d"></a>
# NEUS 642 - Week 10 In-class Exercises
<a href="#Overview">Return to overview</a>


For this week's exercise, we will be utilizing what we learned in week 3. We will be working with more dataframes, creating graphs, and performing statistical analyzes. We will also alter the data set based on predetermined parameters.

<a id="0bc2e68b-c3ef-4061-a613-f7e4f0d6cdb2"></a>
## Background
<a href="#Overview">Return to overview</a>

My project focuses on investigating the effects of hypertension on the heart over time. Angiotensin II (AngII) is a key mediator of the Renin-Angiontensin-Aldosterone system (RAAS) that contributes to hypertension by causing vasoconstriction and increasing sodium reabsorption. We infused our mice with AngII via a minipump for 2, 5, 7, 10, 14, and 28 days. We validate the AngII by measuring the mice's blood pressure and assess functional and structural changes in the heart. Functional changes were measured via an echocardiogram. We specifically measure the ejection fraction, which is the percentage of blood being pushed out of the heart during each contraction.

<center><img src="Heart Diagram.png" width="400"/></center>

We assess structural changes by measuring the sympathetic nerve density using immunohistochemistry for tyrosine hydroxylase (TH). In the image above, each of the rectangles represent an image taken. We take about 3-4 images per area of the heart: left ventricle (LV), intraventricular septum (IVS), and right ventricle (RV). The LV is further divded into the subepicardium (Epi), the outer layer of the heart, and the subendocardium (Endo), the inner layer. The data is first measured as the percentage of TH positive (TH+) pixels. TWe will need to normalize them to the saline control. Below is an example image of the sympathetic nerve density. 

<center><img src="Saline_LV.png" width="400"/></center>

<a id="a547739b-e2ac-49e0-ab7d-7541d1ed025d"></a>
## Getting Started
<a href="#Overview">Return to overview</a>

Let's start by importing our libraries and data `HTN_log`.

In [ ]:
import numpy as np
import pandas as pd
pd.options.display.max_rows = 7
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
import itertools

In [ ]:
HTN_data = pd.read_csv('HTN_log.csv')
HTN_data

<a id="a12e372b-0b50-47cf-bb12-c9f4f606aff1"></a>
# Exercise 1
<a href="#Overview">Return to overview</a>

First, we will grouped the data based on what `Treatment` they recieved. Also display the mean of each group. 

In [ ]:
%load "answers/answer_001.txt"

The `Treatment` is in alphabetical order, but, in our case, this is not useful. Try rearranging the order of the `Treatment` group based chronological order, starting with our `Saline` control using `pd.Categorical`. Redo the grouping and verify that they are in the correct order

In [ ]:
%load "answers/answer_002.txt"

<a id="f84ebf64-6f5a-4fe9-8101-f885b152023e"></a>
# Exercise 2
<a href="#Overview">Return to overview</a>

Since this study is focused on the effects of hypertension, let's start by visualizing the blood pressure (`BP`) data first. For this study, we define hypertension has having a mean arterial pressure (MAP) of at least 120 mmHg. Create a bar graph and include a line at the threshold. Also, include an error bard of the standard deviation. Don't forget to label the axis and add color.

In [ ]:
%load "answers/answer_003.txt"

While we can see that all of the groups treated with AngII are above our threshold, a bar graph alone is not a good representation of the variability in the data. We can see based on the error bars that there are some mice treated with AngII that are below the threshold. Let's add each individual point to our graphs. 

In [ ]:
# Copy and paste your graph here
plt.bar(mean.index, mean.values, yerr = stds, capsize = 5,
        color = colors, edgecolor = 'black')
plt.xlabel('Treatment Type')
plt.ylabel('MAP (mmHg)')
plt.title('Mean BP by Treatment Type')
plt.axhline(y=120, linestyle='--', color = 'black')

# Overlay individual points
for i, treatment in enumerate(mean.index):
    y = HTN_data.loc[HTN_data['Treatment'] == treatment, 'BP']
    
    # Add jitter so points don't stack perfectly
    x = np.random.normal(i, 0.15, size=len(y))
    plt.scatter(x, y, color='black', alpha=1)

Now that we can see each individual score, we can see that several mice treated with AngII did not have hypertention. This is most likely due to an issue with the minipump administering the drug. In order for our results to be more accurate, we need to remove any mice with a MAP less than 120 with the exception of the saline controls and the mice treated for only 2 days. 

In [ ]:
tx_to_filter = ['5 days', '7 days', '10 days', '14 days', '28 days']

filtered = HTN_data[~((HTN_data['Treatment'].isin(tx_to_filter)) &
        (HTN_data['BP'] < 120))].copy()
mean = filtered.groupby('Treatment', observed = True)['BP'].mean()
stds = filtered.groupby('Treatment', observed = True)['BP'].std()

Afer removing the non-hypertensive mice, let's remake the graph. 

In [ ]:
plt.bar(mean.index, mean.values,  yerr = stds, capsize = 5,
        color = colors, edgecolor = 'black')
plt.xlabel('Treatment Type')
plt.ylabel('MAP (mmHg)')
plt.title('Mean BP by Treatment Type')
plt.axhline(y=120, linestyle='--', color = 'black')

# Overlay individual points
for i, treatment in enumerate(mean.index):
    y = filtered.loc[filtered['Treatment'] == treatment, 'BP']
    x = np.random.normal(i, 0.12, size=len(y))
    plt.scatter(x, y, color='black', alpha=1)

All of our individual points should be above the threshold for the mice treated with AngII post-5 days.

<a id="70c0aa42-a1c0-4e48-be53-c406565e64db"></a>
## Exercise 3
<a href="#Overview">Return to overview</a>

Now that our data looks the way we want, let's perform some statisics. We will conduct a one-way ANOVA comparing the mean of each treatment group to the saline control. 

In [ ]:
groups = [
    filtered.loc[filtered['Treatment'] == t, 'BP']
    for t in filtered['Treatment'].unique()]

f_stat, p_val = stats.f_oneway(*groups)

print(f"ANOVA: F={f_stat:.3f}, p={p_val:.4f}")

In [ ]:
results = []

for group in treatment_order:
    if group == 'Saline':
        continue
    d_control = filtered.loc[filtered['Treatment'] == 'Saline', 'BP'].dropna()
    d_group   = filtered.loc[filtered['Treatment'] == group, 'BP'].dropna()
    if len(d_control) >= 2 and len(d_group) >= 2:
        t_stat, p = stats.ttest_ind(d_control, d_group, equal_var=False)
        results.append((group, p))

# Holm Correction
p_vals = [r[1] for r in results]

reject, p_corrected, _, _ = multipletests(p_vals, method='holm')

corrected_results = [
    (results[i][0], p_corrected[i])
    for i in range(len(results))]

print("Comparisons vs Saline:")
for group, p in corrected_results:
    print(f"Saline vs {group}: p = {p:.4f}")

Now that we have the p-values for each group, let's add them to our graph.

In [ ]:
plt.bar(mean.index, mean.values,  yerr = stds, capsize = 5,
        color = colors, edgecolor = 'black')
plt.xlabel('Treatment Type')
plt.ylabel('MAP (mmHg)')
plt.title('Mean BP by Treatment Type')
plt.axhline(y=120, linestyle='--', color = 'black')

# Overlay individual points
for i, treatment in enumerate(mean.index):
    y = filtered.loc[filtered['Treatment'] == treatment, 'BP']
    x = np.random.normal(i, 0.12, size=len(y))
    plt.scatter(x, y, color='black', alpha=1)

#Adding significance bars
control_index = 0

y_max = max(mean.values + stds.values)
height_step = y_max * 0.08
current_height = y_max + height_step

for group, p in corrected_results:
    if p < 0.05:
        group_index = treatment_order.index(group)

        # Draw bracket
        plt.plot(
            [control_index, control_index, group_index, group_index],
            [current_height,
             current_height + height_step,
             current_height + height_step,
             current_height],
            c='black')

        # Convert p-value to stars
        if p < 0.001:
            stars = '***'
        elif p < 0.01:
            stars = '**'
        else:
            stars = '*'

        plt.text(
            (control_index + group_index) / 2,
            current_height + height_step,
            stars,
            ha='center')

        current_height += height_step

As we would expect, all of the mice treated with AngII are significantly different from our saline control. Now, let's visualize any functional changes in the heart. As a reminder, cardiac function was measured using an echocardiogram. More specifically, we use the Ejection Fraction (`EF`), which is the percentage of blood that the heart pushes out during each contration, to represent cardiac function. 

In [ ]:
%load "answers/answer_004.txt"

In [ ]:
%load "answers/answer_005.txt"

Based on the graph, there appears to be a significant decline in cardiac function in mice treated with AngII at 7 and 28 days. We know that the mice are experiencing heart failure at 28 days, but it is unlike that the mice at 7 days are. Let's see if there are structural changes that may explain the decline in function.

<a id="13ca54dd-169d-4632-ab4e-60b4ab2aff9c"></a>
## Exercise 4
<a href="#Overview">Return to overview</a>

Once again, structural changes were determined using the sympathetic nerve density via IHC. Currently, the data is measured as the percentage of TH+ fibers. Before we perform our statistic and make our graph, we want to normalize the data to the `Saline` control. Overwrite and normalized the following columns: `IVS`, `LV`, `RV`, `Epi`, and `Endo`.

In [ ]:
%load "answers/answer_006.txt"

Now that we've normalized the IHC data, let's take a look at it. Unlike the previous stats, this time, we will be comparing the mean of each group to one another, starting with the `LV` (note the `location` variable). 

In [ ]:
location = 'LV'
results = []

for g1, g2 in itertools.combinations(treatment_order, 2):

    d1 = filtered.loc[
        filtered['Treatment'] == g1, location
    ].dropna()

    d2 = filtered.loc[
        filtered['Treatment'] == g2, location
    ].dropna()

    if len(d1) >= 2 and len(d2) >= 2:
        t_stat, p = stats.ttest_ind(
            d1,
            d2,
            equal_var=False) # Welch

        results.append((g1, g2, p))

# Simple Bonferroni Correction
m = len(results)
corrected_results = []

for g1, g2, p in results:
    p_corrected = min(p * m, 1.0)
    corrected_results.append((g1, g2, p_corrected))

# Print Results
print("Pairwise Comparisons", location, "\n")
print(f"{'Group 1':<12} {'Group 2':<12} {'p':<12} {'p_corrected':<15} {'Significant'}")

for (g1, g2, p), (_, _, p_corr) in zip(results, corrected_results):

    sig = "YES" if p_corr < 0.05 else "NO"

    print(f"{g1:<12} {g2:<12} {p:<12.4f} {p_corr:<15.4f} {sig}")

In [ ]:
mean = filtered.groupby('Treatment', observed = True)[location].mean()
stds = filtered.groupby('Treatment', observed = True)[location].std()

colors = plt.cm.tab10(range(len(grouped)))
plt.bar(mean.index, mean.values,  yerr = stds, capsize = 5,
        color = colors, edgecolor = 'black')
plt.xlabel('Treatment Type')
plt.ylabel('Normalized % TH+ fibers')
plt.title('Sympathetic Nerve Density Across Time')

# Overlay individual points
for i, treatment in enumerate(mean.index):
    y = filtered.loc[filtered['Treatment'] == treatment, location]
    x = np.random.normal(i, 0.12, size=len(y))
    plt.scatter(x, y, color='black', alpha=1)
x = np.arange(len(treatment_order))

# Significance bar
y_max = max(mean.values + stds.values)
height_step = y_max * 0.08
current_height = y_max + height_step

for g1, g2, p in corrected_results:
    if p < 0.05:
        i = treatment_order.index(g1)
        j = treatment_order.index(g2)

        x1, x2 = min(i, j), max(i, j)

        plt.plot(
            [x1, x1, x2, x2],
            [current_height,
             current_height + height_step,
             current_height + height_step,
             current_height],
            c='black')

        if p < 0.001:
            stars = '***'
        elif p < 0.01:
            stars = '**'
        else:
            stars = '*'

        plt.text(
            (x1 + x2) / 2,
            current_height + height_step,
            stars,
            ha='center')

        current_height += height_step

Replace the `location` variable with any of the other areas in the heart. From this data, we can see that the sympathetic nerve density is depleted in the mice treated with AngII for 7 days. This lost of sympathetic nerves can explain the lost of cardiac function. 